# FIFA World Cup 2026 Tournament Simulator - Colab Training Notebook

Use this notebook to train the models in Google Colab, save artifacts to Google Drive, and keep the dataset organized in the expected project paths.

## What this notebook does

1. Mounts Google Drive
2. Clones the project repository
3. Installs Python dependencies
4. Prepares the expected dataset folders
5. Trains the match prediction models
6. Saves model artifacts and metadata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

PROJECT_DIR = Path('/content/world-cup-2026-simulator')
DRIVE_DIR = Path('/content/drive/MyDrive/world-cup-2026-simulator')
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
DRIVE_DATA_DIR = DRIVE_DIR / 'data' / 'raw'

In [ ]:
# Clone the repository into the Colab runtime if it is not already present.
if not PROJECT_DIR.exists():
    !git clone https://github.com/your-username/world-cup-2026-simulator.git /content/world-cup-2026-simulator

%cd /content/world-cup-2026-simulator

In [ ]:
# Install dependencies.
!pip install -r requirements.txt

## Download the dataset from Kaggle

This notebook can download the main historical results dataset directly from Kaggle.

Before running the next cell, upload your `kaggle.json` API token from Kaggle. In Colab, you can either:

1. Upload `kaggle.json` from your machine
2. Store it in Google Drive and copy it into place

The cell below downloads the international football results dataset into `data/raw/`. If you already copied the files from Drive, you can skip it.


In [ ]:
# Download the Kaggle dataset into the expected raw data folder.
!pip -q install kaggle

from pathlib import Path
import json
import os
import shutil

RAW_DIR = Path('/content/world-cup-2026-simulator/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Option A: upload kaggle.json manually into Colab, then run this cell.
# Option B: store kaggle.json in Google Drive and copy it here before running.
kaggle_token = Path('/content/kaggle.json')
if not kaggle_token.exists():
    drive_token = Path('/content/drive/MyDrive/kaggle.json')
    if drive_token.exists():
        shutil.copy2(drive_token, kaggle_token)

if kaggle_token.exists():
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/kaggle.json
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d martj42/international-football-results-from-1872-to-2017 -p /content/world-cup-2026-simulator/data/raw --unzip
    print('Kaggle dataset downloaded successfully.')
else:
    print('kaggle.json not found. Upload it to Colab or place it in Google Drive at MyDrive/kaggle.json.')

## Dataset locations

The code expects these files inside `data/raw/`:

- `international_results.csv` or `matches.csv`
- `fifa_rankings.csv` or `rankings.csv`

If you already stored the files in Google Drive, the next cell will copy them into the project. If you do not have them yet, the training pipeline will fall back to a synthetic dataset so the notebook still runs end to end.

In [ ]:
# Prepare the expected dataset directory.
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA_DIR.exists():
    for filename in ['international_results.csv', 'matches.csv', 'fifa_rankings.csv', 'rankings.csv']:
        source_file = DRIVE_DATA_DIR / filename
        if source_file.exists():
            shutil.copy2(source_file, DATA_DIR / filename)
            print(f'Copied {filename} from Drive into the project data folder.')
else:
    print('No dataset folder found in Google Drive yet.')
    print('You can upload the CSV files later or let the project use the synthetic fallback dataset.')

In [ ]:
# Optional: inspect the files now available in data/raw.
import os
print(os.listdir(DATA_DIR))

In [ ]:
# Train the model. This will automatically use real CSVs if they exist, otherwise synthetic data.
!python -m src.train_model

## Save artifacts to Google Drive

After training, the model and metadata are stored in `models/`. The next cell copies them to Google Drive so you can reuse them later without retraining.

In [ ]:
# Persist trained artifacts to Drive.
DRIVE_MODELS_DIR = DRIVE_DIR / 'models'
DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for filename in ['match_prediction_model.joblib', 'match_prediction_metadata.json']:
    source_file = PROJECT_DIR / 'models' / filename
    if source_file.exists():
        shutil.copy2(source_file, DRIVE_MODELS_DIR / filename)
        print(f'Saved {filename} to Google Drive.')
    else:
        print(f'{filename} was not found in the local models folder.')

## Quick prediction test

Run a sample prediction to confirm the model loaded correctly.

In [ ]:
from src.predict import MatchPredictor

predictor = MatchPredictor()
prediction = predictor.predict_match('France', 'Japan')
prediction.probabilities.as_dict(), prediction.expected_home_goals, prediction.expected_away_goals

## Notes

- Replace `your-username` in the clone URL with your GitHub username or fork URL.
- If you have the Kaggle dataset, upload the CSV files into `data/raw/` or copy them into your Drive folder before training.
- If you want, you can also extend this notebook with tournament simulation and Streamlit export cells.